# Implementing Calibration and Misbehavior Detection

In [5]:
import sys
import cv2
import numpy as np
import pandas as pd
import logging
import os
import time
sys.path.append('../src')  # Adjust path depending on your structure

print("Current Directory:", os.getcwd())

from head_pose_estimator import HeadPoseEstimator
from mediapipe_face_mesh_detector import MediapipeFaceMeshDetector
from read_video_file import get_video_frames


Current Directory: /Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/notebooks


## Step1: Define Calibration Procedure
Calibration Phase:
- Ask the user to look directly at the screen where typical content would be displayed (e.g. center of the screen where exam questions are likely to appear).
- Capture the head pose over a few seconds to average out any minor movements and determine a baseline "neutral" head pose.

In [15]:
HEAD_POSE_ESTIMATOR_MODEL = "../src/models/head_pose_model/LGBM_finetune_v_optuna.pkl"
face_mesh_detector = MediapipeFaceMeshDetector()
head_pose_estimator = HeadPoseEstimator(model_path=HEAD_POSE_ESTIMATOR_MODEL, vertical_threshold=0.5, horizontal_threshold=0.5)

print("...Loaded models")

...Loaded models


In [23]:
VIDEO_PATH = {
    "center": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/center.mov",
    "left_upper_corner": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/left_upper_corner.mov",
    "right_upper_corner": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/right_upper_corner.mov",
    "right_bottom_corner": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/right_bottom_corner.mov",
    "left_bottom_corner": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/left_bottom_corner.mov",
    "up_side": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/up_side.mov",
    "right_side": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/right_side.mov",
    "left_side": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/left_side.mov",
    "bottom_side": "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/bottom_side.mov",
}

OOF_PATH = "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/outside_screen.mov"

In [4]:
def capture_baseline_head_pose(video_path):
    poses = []
    print("Capturing baseline head pose...")

    for frame in get_video_frames(video_path):
        if frame is not None:
            frame = cv2.flip(frame, 1)
            image_h, image_w, _ = frame.shape
            
            mediapipe_results = face_mesh_detector.detect_landmarks(frame)
            face_features = head_pose_estimator.extract_head_pose_features(mediapipe_results)
            
            if len(face_features) == 0:
                continue
            face_features_df = pd.DataFrame([face_features], columns=head_pose_estimator.feature_columns)
            face_features_normalized_df = head_pose_estimator.normalize(face_features_df)
            head_pose = head_pose_estimator.predict_head_pose(face_features_normalized_df)
            nose_x = face_features_df["nose_x"].values[0] * image_w
            nose_y = face_features_df["nose_y"].values[0] * image_h
            frame = head_pose_estimator.draw_axes(frame, head_pose, nose_x, nose_y)
            pitch, yaw, roll = head_pose["pitch"], head_pose["yaw"], head_pose["roll"]
            poses.append([pitch, yaw, roll])
            
            cv2.putText(frame, f'Pitch: {head_pose["pitch"]:.2f}', (25, 125), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Yaw: {head_pose["yaw"]:.2f}', (25, 175), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Roll: {head_pose["roll"]:.2f}', (25, 225), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Classify: {head_pose["classify"]}', (25, 275), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
            cv2.imshow("Frame", frame)

            if cv2.waitKey(25) == 27:  # Exit on pressing the ESC key
                break
        else:
            break
    cv2.destroyAllWindows()
    
    poses = np.array(poses)
    average_pose = np.mean(poses, axis=0)
    print(f"Average Pose of {video_path}: {average_pose}")
    return average_pose

In [5]:
results = {}
for key, video_path in VIDEO_PATH.items():
    print(f"Captured baseline head pose for {key} video")
    results[key] = capture_baseline_head_pose(video_path)

print("Results:", results)

Captured baseline head pose for center video
Capturing baseline head pose...
Average Pose of /Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/center.mov: [-0.09226398  0.01817366 -0.06828834]
Captured baseline head pose for left_upper_corner video
Capturing baseline head pose...
Average Pose of /Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/left_upper_corner.mov: [-0.05269493  0.03017593 -0.0832222 ]
Captured baseline head pose for right_upper_corner video
Capturing baseline head pose...
Average Pose of /Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/right_upper_corner.mov: [-0.00465508 -0.16647189 -0.11742412]
Captured baseline head pose for right_bottom_corner video
Capturing baseline head pose...
Average Pose of /Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/dat

In [6]:
results

{'center': array([-0.09226398,  0.01817366, -0.06828834]),
 'left_upper_corner': array([-0.05269493,  0.03017593, -0.0832222 ]),
 'right_upper_corner': array([-0.00465508, -0.16647189, -0.11742412]),
 'right_bottom_corner': array([-0.08143655, -0.1067849 ,  0.01398191]),
 'left_bottom_corner': array([ 0.00094958,  0.0726428 , -0.13246023]),
 'up_side': array([-0.02415382, -0.06999738, -0.10161713]),
 'right_side': array([-0.14343228,  0.14520423,  0.0191516 ]),
 'left_side': array([-0.07485204,  0.01708766, -0.05405652]),
 'bottom_side': array([ 0.09278527, -0.03480367, -0.04376402])}

In [12]:
# Calculate the max/min for pitch and yaw across all points
max_pitch = max([pose[0] for pose in results.values()])
min_pitch = min([pose[0] for pose in results.values()])
max_yaw = max([pose[1] for pose in results.values()])
min_yaw = min([pose[1] for pose in results.values()])

print(f"Max Pitch: {max_pitch}, Min Pitch: {min_pitch}")
print(f"Max Yaw: {max_yaw}, Min Yaw: {min_yaw}")

# Setting thresholds based on the range observed
pitch_threshold = (max_pitch - min_pitch) * 0.75
yaw_threshold = (max_yaw - min_yaw) * 0.75

print(f"Pitch Threshold: {pitch_threshold}, Yaw Threshold: {yaw_threshold}")

Max Pitch: 0.09278527216509898, Min Pitch: -0.14343227521013152
Max Yaw: 0.14520423310107958, Min Yaw: -0.1664718934706329
Pitch Threshold: 0.17716316053142286, Yaw Threshold: 0.23375709492878438


## Capture baseline from webcam

In [26]:
def webcam_capture_baseline_head_pose(duration=5):
    starring_points = ["center", "left_upper_corner", "right_upper_corner", "right_bottom_corner", "left_bottom_corner", "up_side", "right_side", "left_side", "bottom_side"]
    cap = cv2.VideoCapture(0)
    
    all_pose = {}
    
    print("Capturing baseline head pose...")
    
    for starring_point in starring_points:
        poses = []
        start_time = time.time()
        while time.time() - start_time < duration:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.flip(frame, 1)
            image_h, image_w, _ = frame.shape

            mediapipe_results = face_mesh_detector.detect_landmarks(frame)
            face_features = head_pose_estimator.extract_head_pose_features(mediapipe_results)

            if len(face_features) == 0:
                continue
            face_features_df = pd.DataFrame([face_features], columns=head_pose_estimator.feature_columns)
            face_features_normalized_df = head_pose_estimator.normalize(face_features_df)
            head_pose = head_pose_estimator.predict_head_pose(face_features_normalized_df)
            nose_x = face_features_df["nose_x"].values[0] * image_w
            nose_y = face_features_df["nose_y"].values[0] * image_h
            frame = head_pose_estimator.draw_axes(frame, head_pose, nose_x, nose_y)
            pitch, yaw, roll = head_pose["pitch"], head_pose["yaw"], head_pose["roll"]
            poses.append([pitch, yaw, roll])

            cv2.putText(frame, f'Starring point: {starring_point}', (225, 75), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(frame, f'Pitch: {head_pose["pitch"]:.2f}', (25, 125), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Yaw: {head_pose["yaw"]:.2f}', (25, 175), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Roll: {head_pose["roll"]:.2f}', (25, 225), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Classify: {head_pose["classify"]}', (25, 275), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            cv2.imshow("Frame", frame)

            if cv2.waitKey(25) == 27:
                break
            
        all_pose[starring_point] = np.mean(np.array(poses), axis=0)
    cap.release()
    cv2.destroyAllWindows()
    
    return all_pose

In [27]:
results = webcam_capture_baseline_head_pose(duration=10)

Capturing baseline head pose...


In [28]:
results

{'center': array([-0.03967818, -0.0471089 , -0.02070483]),
 'left_upper_corner': array([ 0.08825569, -0.39781565, -0.01866716]),
 'right_upper_corner': array([ 0.21411082,  0.24477476, -0.16741453]),
 'right_bottom_corner': array([-0.03519913,  0.41103205, -0.02903695]),
 'left_bottom_corner': array([-0.10265474, -0.20964889, -0.02915427]),
 'up_side': array([ 0.05191745, -0.04439394, -0.05042405]),
 'right_side': array([ 0.03823496,  0.37968546, -0.09859164]),
 'left_side': array([-0.04892016, -0.11260141, -0.01670708]),
 'bottom_side': array([-0.16776992, -0.02247014, -0.02097178])}

In [29]:
# Calculate the max/min for pitch and yaw across all points
max_pitch = max([pose[0] for pose in results.values()])
min_pitch = min([pose[0] for pose in results.values()])
max_yaw = max([pose[1] for pose in results.values()])
min_yaw = min([pose[1] for pose in results.values()])

print(f"Max Pitch: {max_pitch}, Min Pitch: {min_pitch}")
print(f"Max Yaw: {max_yaw}, Min Yaw: {min_yaw}")

# Setting thresholds based on the range observed
pitch_threshold = (max_pitch - min_pitch) * 0.75
yaw_threshold = (max_yaw - min_yaw) * 0.5

print(f"Pitch Threshold: {pitch_threshold}, Yaw Threshold: {yaw_threshold}")

Max Pitch: 0.21411082473593473, Min Pitch: -0.16776991523189858
Max Yaw: 0.411032049077778, Min Yaw: -0.39781564747498116
Pitch Threshold: 0.286410554975875, Yaw Threshold: 0.40442384827637956


## Test with subjects

### Subject 1

In [48]:
def detect_head_pose_behavior(video_path, pitch_threshold, yaw_threshold, output_path=None):
    head_pose_estimator = HeadPoseEstimator(model_path=HEAD_POSE_ESTIMATOR_MODEL, vertical_threshold=pitch_threshold, horizontal_threshold=yaw_threshold)
    print("Detecting head pose behavior...")

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))  # Get frames per second from the video
    print("FPS:", fps)
    current_frame = 0
    results = []
    
    os.makedirs(output_path, exist_ok=True)

    for frame in get_video_frames(video_path):
        if frame is not None:
            frame = cv2.flip(frame, 1)
            image_h, image_w, _ = frame.shape
            
            mediapipe_results = face_mesh_detector.detect_landmarks(frame)
            face_features = head_pose_estimator.extract_head_pose_features(mediapipe_results)
            
            if len(face_features) == 0:
                continue
            face_features_df = pd.DataFrame([face_features], columns=head_pose_estimator.feature_columns)
            face_features_normalized_df = head_pose_estimator.normalize(face_features_df)
            head_pose = head_pose_estimator.predict_head_pose(face_features_normalized_df)
            nose_x = face_features_df["nose_x"].values[0] * image_w
            nose_y = face_features_df["nose_y"].values[0] * image_h
            frame = head_pose_estimator.draw_axes(frame, head_pose, nose_x, nose_y)
            
            cv2.putText(frame, f'Pitch: {head_pose["pitch"]:.2f}', (25, 125), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Yaw: {head_pose["yaw"]:.2f}', (25, 175), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Roll: {head_pose["roll"]:.2f}', (25, 225), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Classify: {head_pose["classify"]}', (25, 275), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
            cv2.imshow("Frame", frame)

            if cv2.waitKey(25) == 27:  # Exit on pressing the ESC key
                break
            
            if current_frame % fps == 0:
                time_MMSS = time.strftime("%M:%S", time.gmtime(current_frame // fps))
                head_pose["time"] = time_MMSS
                results.append(head_pose)
                cv2.imwrite(f"{output_path}/frame_{current_frame}_{time_MMSS}.jpg", frame)
            
            current_frame += 1
        else:
            break
        
    cv2.destroyAllWindows()
    
    df = pd.DataFrame(results)
    df.to_csv(f"{output_path}/head_pose_results.csv", index=False)
    

In [46]:
def detect_head_pose_behavior_advance(video_path, pitch_threshold, yaw_threshold, start_time, end_time):
    head_pose_estimator = HeadPoseEstimator(model_path=HEAD_POSE_ESTIMATOR_MODEL, vertical_threshold=pitch_threshold, horizontal_threshold=yaw_threshold)
    print("Detecting head pose behavior...")

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))  # Get frames per second from the video
    print("FPS:", fps)
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)
    current_frame = 0

    for frame in get_video_frames(video_path):
        if frame is not None:
            if current_frame < start_frame:
                current_frame += 1
                continue

            if current_frame > end_frame:
                break

            frame = cv2.flip(frame, 1)
            image_h, image_w, _ = frame.shape

            mediapipe_results = face_mesh_detector.detect_landmarks(frame)
            face_features = head_pose_estimator.extract_head_pose_features(mediapipe_results)

            if len(face_features) == 0:
                current_frame += 1
                continue
            face_features_df = pd.DataFrame([face_features], columns=head_pose_estimator.feature_columns)
            face_features_normalized_df = head_pose_estimator.normalize(face_features_df)
            head_pose = head_pose_estimator.predict_head_pose(face_features_normalized_df)
            nose_x = face_features_df["nose_x"].values[0] * image_w
            nose_y = face_features_df["nose_y"].values[0] * image_h
            frame = head_pose_estimator.draw_axes(frame, head_pose, nose_x, nose_y)

            cv2.putText(frame, f'Pitch: {head_pose["pitch"]:.2f}', (25, 125), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Yaw: {head_pose["yaw"]:.2f}', (25, 175), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Roll: {head_pose["roll"]:.2f}', (25, 225), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, f'Classify: {head_pose["classify"]}', (25, 275), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            cv2.imshow("Frame", frame)
            if cv2.waitKey(25) == 27:  # Exit on pressing the ESC key
                break
            
            # Save the frame every 1 second
            if current_frame % fps == 0:
                time_MMSS = time.strftime("%M_%S", time.gmtime(current_frame // fps))
                cv2.imwrite(f"frame_{current_frame}_{time_MMSS}.jpg", frame)

            current_frame += 1
        else:
            break
    cv2.destroyAllWindows()

In [56]:
def detect_head_pose_behavior_result(video_path, pitch_threshold, yaw_threshold, output_path=None):
    head_pose_estimator = HeadPoseEstimator(model_path=HEAD_POSE_ESTIMATOR_MODEL, vertical_threshold=pitch_threshold, horizontal_threshold=yaw_threshold)
    print("Detecting head pose behavior...")

    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))  # Get frames per second from the video
    print("FPS:", fps)
    current_frame = 0
    results = []
    
    os.makedirs(output_path, exist_ok=True)

    for frame in get_video_frames(video_path):
        if frame is not None:
            # Process every second of video
            if current_frame % fps == 0:
                frame = cv2.flip(frame, 1)
                image_h, image_w, _ = frame.shape
                
                mediapipe_results = face_mesh_detector.detect_landmarks(frame)
                face_features = head_pose_estimator.extract_head_pose_features(mediapipe_results)
                
                if len(face_features) == 0:
                    current_frame += 1
                    none_head_pose = {
                        "pitch": None,
                        "yaw": None,
                        "roll": None,
                        "classify": None,
                        "time": time.strftime("%M:%S", time.gmtime(current_frame // fps))
                    }
                    results.append(none_head_pose)
                    continue
                face_features_df = pd.DataFrame([face_features], columns=head_pose_estimator.feature_columns)
                face_features_normalized_df = head_pose_estimator.normalize(face_features_df)
                head_pose = head_pose_estimator.predict_head_pose(face_features_normalized_df)
                nose_x = face_features_df["nose_x"].values[0] * image_w
                nose_y = face_features_df["nose_y"].values[0] * image_h
                frame = head_pose_estimator.draw_axes(frame, head_pose, nose_x, nose_y)
                
                time_MMSS = time.strftime("%M:%S", time.gmtime(current_frame // fps))
                head_pose["time"] = time_MMSS
                results.append(head_pose)
                
                cv2.putText(frame, f'Pitch: {head_pose["pitch"]:.2f}', (25, 125), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                cv2.putText(frame, f'Yaw: {head_pose["yaw"]:.2f}', (25, 175), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                cv2.putText(frame, f'Roll: {head_pose["roll"]:.2f}', (25, 225), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                cv2.putText(frame, f'Classify: {head_pose["classify"]}', (25, 275), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                cv2.putText(frame, f"Time: {time_MMSS}", (25, 325), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                
                cv2.imwrite(f"{output_path}/frame_{current_frame}_{time_MMSS}.jpg", frame)

            current_frame += 1
        else:
            break
    
    df = pd.DataFrame(results)
    df.to_csv(f"{output_path}/head_pose_results.csv", index=False)

In [52]:
SUBJECT_VIDEO_PATH = "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/processed/OEP_dataset/subject1/Yousef1.mp4"

In [55]:
detect_head_pose_behavior_result(SUBJECT_VIDEO_PATH, pitch_threshold=0.286410554975875, yaw_threshold=0.40442384827637956, output_path="headpose/subject1")

Detecting head pose behavior...
FPS: 25


In [32]:
detect_head_pose_behavior(SUBJECT_VIDEO_PATH, pitch_threshold=0.286410554975875, yaw_threshold=0.40442384827637956)

Detecting head pose behavior...


### Subject 3

In [34]:
SUBJECT_VIDEO_PATH = "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/processed/OEP_dataset/subject3/XiYin1.mp4"

In [36]:
detect_head_pose_behavior(SUBJECT_VIDEO_PATH, pitch_threshold=0.286410554975875, yaw_threshold=0.40442384827637956)

Detecting head pose behavior...


### Subject Gritt

In [37]:
SUBJECT_VIDEO_PATH = "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/gritt_exam.mov"

In [50]:
detect_head_pose_behavior(SUBJECT_VIDEO_PATH, pitch_threshold=0.286410554975875, yaw_threshold=0.40442384827637956, output_path="headpose/gritt_exam")

Detecting head pose behavior...
FPS: 30


In [47]:
detect_head_pose_behavior_advance(SUBJECT_VIDEO_PATH, pitch_threshold=0.286410554975875, yaw_threshold=0.40442384827637956, start_time=307, end_time=370)

Detecting head pose behavior...
FPS: 30
